In [ ]:
!pip install --upgrade sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 14.7 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [ ]:
import os
import json
import pickle
import torch

from google.colab import drive
from sentence_transformers import SentenceTransformer


# ======================================================================
# CONFIGURATION
# ======================================================================

MODEL_NAME = "BAAI/bge-large-en-v1.5"
DEVICE = "cuda"
BATCH_SIZE = 64

SOURCE_DIR = "/content/drive/MyDrive/Hadolint_Data"
OUTPUT_DIR = os.path.join(SOURCE_DIR, "embeddings_cache")


# ======================================================================
# GOOGLE DRIVE
# ======================================================================

print("=" * 70)
print("🚀 STRUCTURED EMBEDDING GENERATION")
print("=" * 70)

print("\n📂 Mounting Google Drive...")
drive.mount("/content/drive")


# ======================================================================
# GPU VALIDATION
# ======================================================================

print("\n🧪 Checking hardware...")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ CRITICAL ERROR: CUDA GPU was not detected.\n"
        "Go to Runtime > Change runtime type and select a T4 GPU."
    )

print(
    f"   🚀 GPU detected: "
    f"{torch.cuda.get_device_name(0)}"
)


# ======================================================================
# MODEL
# ======================================================================

print("\n🧠 Loading embedding model...")
print(f"   Model: {MODEL_NAME}")

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

print("   ✅ Model loaded successfully.")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")


# ======================================================================
# TEXT REPRESENTATIONS
# ======================================================================

def build_hadolint_embedding_text(rule):
    """
    Build the textual representation used for Hadolint embeddings.

    Representation:

        ID
        TITLE
        PROBLEMATIC CODE
        CORRECT CODE

    Rationale is deliberately excluded.
    """

    parts = []

    rule_id = rule.get("id")
    title = rule.get("title")
    problematic_code = rule.get("problematic_code")
    correct_code = rule.get("correct_code")

    if rule_id:
        parts.append(str(rule_id))

    if title:
        parts.append(str(title))

    if problematic_code:
        parts.append(
            "Problematic code:\n" + str(problematic_code)
        )

    if correct_code:
        parts.append(
            "Correct code:\n" + str(correct_code)
        )

    return "\n\n".join(parts).strip()


def build_iec_embedding_text(requirement):
    """
    Build the textual representation used for IEC 62443 embeddings.

    Representation:

        ID
        TITLE
        NORMATIVE TEXT

    Rationale is deliberately excluded.
    """

    parts = []

    requirement_id = requirement.get("id")
    title = requirement.get("title")
    text = requirement.get("text")

    if requirement_id:
        parts.append(str(requirement_id))

    if title:
        parts.append(str(title))

    if text:
        parts.append(str(text))

    return "\n\n".join(parts).strip()


# ======================================================================
# DATASET LOADERS
# ======================================================================

def load_structured_dataset(file_path, dataset_name):
    """
    Load either of the two structured dataset formats used
    in the project.

    Hadolint:

        {
            "DL1001": {...},
            "DL3000": {...}
        }

    IEC:

        {
            "metadata": {...},
            "requirements": [...]
        }

    Returns:
        metadata, requirements
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Dataset not found: {file_path}"
        )

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        raise ValueError(
            f"Invalid dataset format in {file_path}: "
            "root must be a JSON object."
        )

    # --------------------------------------------------------------
    # IEC 62443 STRUCTURE
    # --------------------------------------------------------------

    if "requirements" in data:

        requirements = data["requirements"]

        if not isinstance(requirements, list):
            raise ValueError(
                f"Invalid IEC dataset format in {file_path}: "
                "'requirements' must be a list."
            )

        metadata = data.get("metadata", {})

        return metadata, requirements

    # --------------------------------------------------------------
    # HADOLINT STRUCTURE
    # --------------------------------------------------------------

    if dataset_name.lower() == "hadolint":

        requirements = []

        for rule_id, rule in data.items():

            if not isinstance(rule, dict):
                continue

            # Ensure the ID exists even if the JSON key is
            # the only place where it is defined.
            if not rule.get("id"):
                rule["id"] = rule_id

            requirements.append(rule)

        metadata = {
            "dataset": "hadolint",
            "source_format": "id-indexed dictionary"
        }

        return metadata, requirements

    # --------------------------------------------------------------
    # UNKNOWN FORMAT
    # --------------------------------------------------------------

    raise ValueError(
        f"Unsupported dataset structure in {file_path}. "
        "Expected either an IEC 'requirements' list or "
        "a Hadolint ID-indexed dictionary."
    )


def generate_embeddings(
    dataset_name,
    input_file,
    output_file,
    text_builder
):
    """
    Generate normalized embeddings for a structured dataset.

    Output format:

    {
        "metadata": {
            ...
        },
        "embeddings": {
            "RULE_ID": vector
        }
    }
    """

    print("\n" + "=" * 70)
    print(f"📚 Processing {dataset_name.upper()}")
    print("=" * 70)

    print("📄 Input:")
    print(f"   {input_file}")

    metadata, requirements = load_structured_dataset(
        input_file,
        dataset_name
    )

    print(f"   ✅ Loaded {len(requirements)} entries.")

    ids = []
    texts = []

    skipped = 0

    for requirement in requirements:

        if not isinstance(requirement, dict):
            skipped += 1
            continue

        rule_id = requirement.get("id")

        if not rule_id:
            skipped += 1
            continue

        text = text_builder(requirement)

        if not text:
            skipped += 1
            continue

        ids.append(rule_id)
        texts.append(text)

    if skipped:
        print(
            f"   ⚠️ Skipped {skipped} invalid/empty entries."
        )

    if not ids:
        raise ValueError(
            f"No valid entries found in {input_file}"
        )

    # --------------------------------------------------------------
    # ID UNIQUENESS
    # --------------------------------------------------------------

    if len(ids) != len(set(ids)):

        seen = set()
        duplicates = []

        for rule_id in ids:
            if rule_id in seen:
                duplicates.append(rule_id)
            else:
                seen.add(rule_id)

        raise ValueError(
            f"Duplicate IDs found in {dataset_name}: "
            f"{duplicates}"
        )

    print(f"   🔢 Entries to embed: {len(ids)}")

    # --------------------------------------------------------------
    # SAMPLE
    # --------------------------------------------------------------

    print("\n📝 Sample embedding representation:")
    print("-" * 70)
    print(texts[0])
    print("-" * 70)

    # --------------------------------------------------------------
    # GENERATE
    # --------------------------------------------------------------

    print("\n⚡ Generating embeddings on GPU...")

    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    print(
        f"   ✅ Generated matrix: "
        f"{embeddings.shape}"
    )

    # --------------------------------------------------------------
    # CACHE
    # --------------------------------------------------------------

    representation = (
        "id+title+problematic_code+correct_code"
        if dataset_name.lower() == "hadolint"
        else
        "id+title+normative_text"
    )

    cache = {
        "metadata": {
            "dataset": dataset_name,
            "model": MODEL_NAME,
            "dimension": int(
                model.get_embedding_dimension()
            ),
            "normalized": True,
            "representation": representation,
            "source_file": input_file,
            "entry_count": len(ids)
        },
        "embeddings": {
            ids[i]: embeddings[i]
            for i in range(len(ids))
        }
    }

    # --------------------------------------------------------------
    # SAVE
    # --------------------------------------------------------------

    os.makedirs(
        os.path.dirname(output_file),
        exist_ok=True
    )

    with open(output_file, "wb") as f:
        pickle.dump(cache, f)

    print("\n💾 Cache saved:")
    print(f"   {output_file}")

    print(
        f"   📦 {len(ids)} embeddings"
    )

    print(
        f"   📐 Dimension: "
        f"{embeddings.shape[1]}"
    )

    return cache


# ======================================================================
# DATASET PATHS
# ======================================================================

HADOLINT_JSON = os.path.join(
    SOURCE_DIR,
    "hadolint_rules_structured.json"
)

IEC_JSON = os.path.join(
    SOURCE_DIR,
    "iec62443_with_rationale.json"
)


HADOLINT_CACHE = os.path.join(
    OUTPUT_DIR,
    "cache_hadolint_structured.pkl"
)

IEC_CACHE = os.path.join(
    OUTPUT_DIR,
    "cache_iec_structured.pkl"
)


# ======================================================================
# GENERATION
# ======================================================================

print("\n🔥 Starting embedding generation...")

hadolint_cache = generate_embeddings(
    dataset_name="Hadolint",
    input_file=HADOLINT_JSON,
    output_file=HADOLINT_CACHE,
    text_builder=build_hadolint_embedding_text
)

iec_cache = generate_embeddings(
    dataset_name="IEC 62443-3-3",
    input_file=IEC_JSON,
    output_file=IEC_CACHE,
    text_builder=build_iec_embedding_text
)


# ======================================================================
# FINAL VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("🏆 EMBEDDING GENERATION COMPLETED")
print("=" * 70)

print("\n📊 Generated datasets:")

print(
    f"   Hadolint : "
    f"{len(hadolint_cache['embeddings'])} embeddings"
)

print(
    f"   IEC      : "
    f"{len(iec_cache['embeddings'])} embeddings"
)

print(
    f"\n📐 Embedding dimension: "
    f"{hadolint_cache['metadata']['dimension']}"
)

print(
    f"🧠 Model: "
    f"{MODEL_NAME}"
)

print(
    f"📏 Normalized: "
    f"{hadolint_cache['metadata']['normalized']}"
)

print("\n📁 Output files:")

print(
    f"   {HADOLINT_CACHE}"
)

print(
    f"   {IEC_CACHE}"
)

print("\n✅ Ready for local similarity calculation.")

🚀 STRUCTURED EMBEDDING GENERATION

📂 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🧪 Checking hardware...
   🚀 GPU detected: Tesla T4

🧠 Loading embedding model...
   Model: BAAI/bge-large-en-v1.5


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

   ✅ Model loaded successfully.
   Embedding dimension: 1024

🔥 Starting embedding generation...

📚 Processing HADOLINT
📄 Input:
   /content/drive/MyDrive/Hadolint_Data/hadolint_rules_structured.json
   ✅ Loaded 594 entries.
   🔢 Entries to embed: 594

📝 Sample embedding representation:
----------------------------------------------------------------------
DL1001

Please refrain from using inline ignore pragmas # hadolint ignore=DLxxxx.

Problematic code:
# hadolint ignore=DL3006
FROM busybox

Correct code:
FROM busybox
----------------------------------------------------------------------

⚡ Generating embeddings on GPU...


/tmp/ipykernel_529/3879140496.py:65: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

   ✅ Generated matrix: (594, 1024)

💾 Cache saved:
   /content/drive/MyDrive/Hadolint_Data/embeddings_cache/cache_hadolint_structured.pkl
   📦 594 embeddings
   📐 Dimension: 1024

📚 Processing IEC 62443-3-3
📄 Input:
   /content/drive/MyDrive/Hadolint_Data/iec62443_with_rationale.json
   ✅ Loaded 100 entries.
   🔢 Entries to embed: 100

📝 Sample embedding representation:
----------------------------------------------------------------------
SR 1.1

Human user identification and authentication

The control system shall provide the capability to identify and authenticate all human users.
----------------------------------------------------------------------

⚡ Generating embeddings on GPU...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

   ✅ Generated matrix: (100, 1024)

💾 Cache saved:
   /content/drive/MyDrive/Hadolint_Data/embeddings_cache/cache_iec_structured.pkl
   📦 100 embeddings
   📐 Dimension: 1024

🏆 EMBEDDING GENERATION COMPLETED

📊 Generated datasets:
   Hadolint : 594 embeddings
   IEC      : 100 embeddings

📐 Embedding dimension: 1024
🧠 Model: BAAI/bge-large-en-v1.5
📏 Normalized: True

📁 Output files:
   /content/drive/MyDrive/Hadolint_Data/embeddings_cache/cache_hadolint_structured.pkl
   /content/drive/MyDrive/Hadolint_Data/embeddings_cache/cache_iec_structured.pkl

✅ Ready for local similarity calculation.
